In [18]:
import os
import openai
import json
import minsearch
from openai import OpenAI
from tqdm.auto import tqdm

In [ ]:

client = OpenAI()

In [20]:
response = client.chat.completions.create(
    model='gpt-4.1',
    messages=[{"role": "user", "content": "how to find helper information of a function in Jupyter Notebook?"}]
)

In [21]:
response.choices[0].message.content

"In Jupyter Notebook, you can **quickly access the helper information (documentation, docstring, or help) of a function in several convenient ways**:\n\n### 1. Using `?` After the Function Name\n\nType the function name followed by `?` and run the cell:\n\n```python\nlen?\n```\n\nThis will display the function's docstring and help information in a window at the bottom of the notebook.\n\n---\n\n### 2. Using Shift+Tab\n\nMove your cursor inside the parentheses of the function and press `Shift + Tab`.\n\nFor example, type:\n\n```python\nlen()\n```\n\nThen, with the cursor between the parentheses, press **Shift+Tab**.  \n- Pressing **Shift+Tab** multiple times (up to 4) expands the amount of information shown.\n\n---\n\n### 3. Using Python’s `help()` Function\n\nYou can also use the built-in `help()` function:\n\n```python\nhelp(len)\n```\n\nThis will print the help information in the cell output.\n\n---\n\n### 4. Using Double Question Marks `??`\n\nUsing `??` after the function name may 

In [22]:
with open('documents.json','rt') as f:
    docs_raw = json.load(f)

In [24]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [9]:
print(len(documents))
documents[0]

948


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [10]:
index = minsearch.Index(
    text_fields=['question','text','section'],
    keyword_fields=['course']
)

index.fit(documents)

In [10]:
q = 'the course has already started, can I still enroll?'
#q = 'the course just started how do I enroll?'

In [11]:
boost = {'question':3.0, 'section':0.5}

results = index.search(
    query=q,
    filter_dict={'course':'data-engineering-zoomcamp'},
    boost_dict=boost,
    num_results=5
)

In [12]:
results

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 202

In [25]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [13]:
prompt = build_prompt(query=q, search_results=results)

In [26]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [15]:
answer = llm(prompt=prompt)

In [16]:
answer

'Yes, you can still enroll in the course even after it has started. You are eligible to submit homework, but make sure to adhere to the deadlines for the final projects.'

In [ ]:
######################################################################################

In [15]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt


def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content


def search(query, index, question_weight=3.0, section_weight=0.5):
    
    boost = {'question': question_weight, 'section': section_weight}
    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [22]:
query = 'the course has already started, can I still enroll?'
q = 'the course has already started, can I still enroll?'
#q = 'the course just started how do I enroll?'

def rag(query):
    search_results = search(query,index)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [23]:
rag(query)

'Yes, even if the course has already started, you can still enroll and are eligible to submit the homework. However, be mindful of the deadlines for turning in the final projects and avoid leaving everything for the last minute.'

In [ ]:
#########################################################################
# Elastic Search: Production-grade search, big data, log analytics, and real-time monitoring.
# MinSearch: Local prototyping, small datasets, LLM tutorials (e.g., DataTalksClub).
#########################################################################

In [1]:
from elasticsearch import Elasticsearch

In [3]:
es_client = Elasticsearch("http://localhost:9200")

In [4]:
es_client.info()

ObjectApiResponse({'name': '7267c26eeec4', 'cluster_name': 'docker-cluster', 'cluster_uuid': 't4j9npGRTSu2zad7kkPaJw', 'version': {'number': '8.4.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '42f05b9372a9a4a470db3b52817899b99a76ee73', 'build_date': '2022-10-04T07:17:24.662462378Z', 'build_snapshot': False, 'lucene_version': '9.3.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [11]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"

es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [13]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

In [14]:
query = 'I just disovered the course. Can I still join it?'

In [15]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []
    
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [27]:
def rag_els(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [28]:
rag_els(query=query)

'Yes, you can still join the course even if you discovered it after the start date. You are eligible to submit the homework without registering, but be mindful of the deadlines for turning in the final projects.'